In [1]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rprint

# ---------------------------------------------------------------------------
# 1. 环境变量与模型初始化
# ---------------------------------------------------------------------------

# 从 .env 文件中加载环境变量；override=True 表示同名变量以 .env 文件为准
load_dotenv(override=True)

# 从环境变量中读取 DeepSeek 的 API Key 和 API Base 地址
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_BASE = os.getenv("DEEPSEEK_API_BASE")

# 初始化聊天模型：
# model 采用 "provider:model_name" 的格式指定 DeepSeek 的模型
# api_key / base_url 用于覆盖默认配置，指向自定义的 DeepSeek 服务
model = init_chat_model(
      model="deepseek:deepseek-v4-flash",
      api_key=DEEPSEEK_API_KEY,
      base_url=DEEPSEEK_API_BASE,
      extra_body={"thinking": {"type": "disabled"}}  # 关键：永久关闭思考模式
)

In [2]:
from pydantic import BaseModel, Field
from rich import print as rprint


class Movie(BaseModel):
    """电影信息"""

    title: str = Field(description=" 电影标题 ")
    year: int = Field(description=" 上映年份 ")
    director: str = Field(description=" 导演 ")
    rating: float = Field(description=" 评分（ 10 分制） ")


# 设置模型结构化输出
model_with_structure = model.with_structured_output(Movie, include_raw=True)

# 调用模型并获取结构化输出
resp = model_with_structure.invoke(" 给我介绍下电影《星际穿越》 ")

print(type(resp))
rprint(resp)

<class 'dict'>


{
    'raw': AIMessage(
        content='',
        additional_kwargs={'refusal': None},
        response_metadata={
            'token_usage': {
                'completion_tokens': 89,
                'prompt_tokens': 349,
                'total_tokens': 438,
                'completion_tokens_details': None,
                'prompt_tokens_details': {
                    'audio_tokens': None,
                    'cache_write_tokens': None,
                    'cached_tokens': 128,
                    'image_tokens': None,
                    'text_tokens': None
                },
                'prompt_cache_hit_tokens': 128,
                'prompt_cache_miss_tokens': 221
            },
            'model_provider': 'deepseek',
            'model_name': 'deepseek-flash',
            'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6',
            'id': 'd58af4e6-c07a-444b-8a70-d572c3e250ea',
            'finish_reason': 'tool_calls',
            'logprobs': None
        },
        id='lc_run--01a1d0c3-b020-709c-ad9f-f1c7bb87020f-0',
        tool_calls=[
            {
                'name': 'Movie',
                'args': {'title': '星际穿越', 'year': 2014, 'director': '克里斯托弗·诺兰', 'rating': 9.4},
                'id': 'call_00_Kyd3ojzTwioI02qjZuI70268',
                'type': 'tool_call'
            }
        ],
        invalid_tool_calls=[],
        usage_metadata={
            'input_tokens': 349,
            'output_tokens': 89,
            'total_tokens': 438,
            'input_token_details': {'cache_read': 128},
            'output_token_details': {}
        }
    ),
    'parsed': Movie(title='星际穿越', year=2014, director='克里斯托弗·诺兰', rating=9.4),
    'parsing_error': None
}